# ESM-2 Geometric Stability Scaling Experiment

Tests the hypothesis: **Bigger Transformer models = lower geometric stability**

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py` to this Colab session
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies
print("Installing dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas
print("Done!")

In [ ]:
# Configuration

# Choose experiment phase: 'quick', 'full', or 'architecture'
import sys, os
sys.path.insert(0, '.')
PHASE = 'quick'  # Start here!

OUTPUT_BASE = './results/esm2_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    # Quick validation: 2 models, 500 sequences
    'quick': {
        'n_sequences': 500,
        'seq_length': 200,       # amino acids (not DNA bases)
        'models': [
            'facebook/esm2_t6_8M_UR50D',      # 8M params
            'facebook/esm2_t33_650M_UR50D',    # 650M params
        ],
        'batch_size': 16,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],      # fewer perturbations for speed
    },
    # Full scaling sweep: all 6 ESM-2 sizes
    'full': {
        'n_sequences': 10000,
        'seq_length': 200,
        'models': [
            'facebook/esm2_t6_8M_UR50D',      # 8M
            'facebook/esm2_t12_35M_UR50D',    # 35M
            'facebook/esm2_t30_150M_UR50D',   # 150M
            'facebook/esm2_t33_650M_UR50D',   # 650M
            'facebook/esm2_t36_3B_UR50D',     # 3B
            # 'facebook/esm2_t48_15B_UR50D',  # 15B -- needs A100/H100
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Sequences: {config['n_sequences']}")
print(f"Sequence length: {config['seq_length']} amino acids")
print(f"Models: {len(config['models'])}")
print(f"Bootstrap rounds: {config['n_bootstrap']}")

In [ ]:
# Generate Protein Sequences
#
# IMPORTANT: ESM-2 is a PROTEIN language model, not DNA.
# It expects amino acid sequences (20 standard AAs).
# For the scaling experiment, random protein sequences are sufficient
# to test the architectural property.

import numpy as np

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')  # 20 standard amino acids

def generate_protein_sequences(n_sequences, seq_length, seed=320):
    """Generate random protein sequences for ESM-2."""
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(AMINO_ACIDS, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} protein sequences (length={seq_length})")
    print(f"Sample: {sequences[0][:50]}...")
    return sequences

sequences = generate_protein_sequences(config['n_sequences'], config['seq_length'])

In [ ]:
# Protein Perturbation Functions
#
# DNA perturbations (reverse complement, motif shift) don't apply to proteins.
# Instead we use amino acid substitutions at various rates.

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_protein(sequence, mutation_rate, rng):
    """Randomly substitute amino acids at the given rate."""
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [aa for aa in AMINO_ACIDS if aa != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_sequence(sequence):
    """Reverse the amino acid sequence (not biologically meaningful,
    but tests whether the model's geometry is order-sensitive)."""
    return sequence[::-1]


class ProteinPerturbationSuite:
    """Perturbation suite adapted for protein sequences / ESM-2."""

    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse

    def run_all(self, sequences):
        results = {}

        # Amino acid substitutions at various rates
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate * 100)}pct"
            perturbed = [mutate_protein(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name,
                category='substitution',
                sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'Random AA substitutions at {rate*100:.0f}% rate',
            )

        # Sequence reversal (order sensitivity test)
        if self.include_reverse:
            name = 'reverse'
            perturbed = [reverse_sequence(s) for s in sequences]
            results[name] = PerturbedSet(
                name=name,
                category='reverse',
                sequences=perturbed,
                params={},
                description='Full sequence reversal',
            )

        return results

    def get_perturbation_names(self):
        names = [f"aa_sub_{int(r*100)}pct" for r in self.snp_rates]
        if self.include_reverse:
            names.append('reverse')
        return names

    def summary(self):
        lines = [
            'Protein Perturbation Protocol',
            '=' * 40,
            f'AA substitution rates: {self.snp_rates}',
            f'Sequence reversal: {self.include_reverse}',
            f'Total perturbation sets: {len(self.get_perturbation_names())}',
        ]
        return '\n'.join(lines)


# Quick test
suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print(suite.summary())

test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# Model Wrapper with GPU Cleanup

import torch
import gc
from transformers import AutoTokenizer, AutoModel


def cleanup_gpu():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_esm2_embedding_fn(model_name, batch_size=8):
    """Load ESM-2 model and return an embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=1024,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens)

            # Mean pool over sequence (excluding special tokens)
            hidden = outputs.last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(pooled.cpu().numpy())

        print()  # newline after progress
        return np.concatenate(embeddings, axis=0)

    n_params = n_params  # already computed above
    return embed, model, tokenizer, n_params


print("Model wrapper ready")

In [ ]:
# Set Up Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,         # ESM-2 embeddings are already pooled
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'ESM-2 SCALING EXPERIMENT -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    # Load model
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_esm2_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    # Generate perturbations
    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    # Compute clean embeddings (batched, with caching)
    safe_name = model_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    # Compute perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    # Free GPU memory before running Shesha (CPU-only)
    del model_obj, tokenizer_obj
    cleanup_gpu()

    # Run Shesha evaluation
    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    # Collect per-perturbation rows
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')
    print(f'  Pert. Magnitude:     {summary["mean_perturbation_magnitude"]:.4f}')

# Save detailed per-perturbation CSV
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/esm2_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Compare Models & Save Results

from evaluation_harness import compare_models
import json

print('Generating comparison...')
comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    short = model_name.split('/')[-1]
    print(f'{short}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print(f'  RDM Similarity:       {s["mean_rdm_similarity_score"]:.4f} +/- {s["std_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:      {s["mean_perturbation_stability_score"]:.4f} +/- {s["std_perturbation_stability_score"]:.4f}')
    print(f'  Pert. Magnitude:      {s["mean_perturbation_magnitude"]:.4f} +/- {s["std_perturbation_magnitude"]:.4f}')
    print()

In [ ]:
# Visualize -- Stability vs Sequence Length

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

# Extract data
summaries = [r.summary() for r in reports]
model_names = [r.model_name for r in reports]

SIZE_MAP = {'8M': 8, '35M': 35, '150M': 150, '650M': 650, '3B': 3000, '15B': 15000}

def extract_size(name):
    for key, val in SIZE_MAP.items():
        if key in name:
            return val
    return 0

sizes = [extract_size(n) for n in model_names]

# Metrics to plot
metrics = {
    'mean_composite_stability': ('Composite Stability', 'tab:blue'),
    'mean_rdm_similarity_score': ('RDM Similarity (clean vs. perturbed)', 'tab:green'),
    'mean_perturbation_stability_score': ('Perturbation Direction Consistency', 'tab:purple'),
    'mean_perturbation_magnitude': ('Perturbation Magnitude', 'tab:red'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (metric_key, (label, color)) in enumerate(metrics.items()):
    ax = axes[idx]
    values = [s[metric_key] for s in summaries]

    # Error bars if available
    std_key = metric_key.replace('mean_', 'std_')
    errors = [s.get(std_key, 0) for s in summaries]

    ax.errorbar(sizes, values, yerr=errors, fmt='o-', color=color,
                linewidth=2, markersize=10, capsize=5, capthick=2)
    ax.set_xscale('log')
    ax.set_xlabel('Parameter Count (millions)')
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}B' if x >= 1000 else f'{x:.0f}M'
    ))

    # Annotate points
    for i, name in enumerate(model_names):
        short = name.split('/')[-1].split('_')[1]  # e.g. 't6', 't33'
        ax.annotate(short, (sizes[i], values[i]),
                    textcoords='offset points', xytext=(0, 12),
                    ha='center', fontsize=9, color='gray')

fig.suptitle(
    'The Scaling Law of Instability: ESM-2 Geometric Stability vs. Model Size',
    fontsize=15, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/esm2_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{RESULTS_DIR}/esm2_scaling_{PHASE}.pdf', bbox_inches='tight')
plt.show()

print(f'Saved to {RESULTS_DIR}/esm2_scaling_{PHASE}.png and .pdf')

In [ ]:
# Per-Perturbation Breakdown
#
# Shows how each perturbation type affects stability across model sizes.
# Look for: do larger models degrade more on specific perturbation types?

# df_detailed was already saved above; display it here
print('Per-perturbation results:')
print(df_detailed.to_string(index=False))

# Also create the df variable for the heatmap cell
df = df_detailed
print(f'\nDetailed CSV at: {RESULTS_DIR}/esm2_scaling_{PHASE}_detailed.csv')

In [ ]:
# Per-Perturbation Heatmap

import seaborn as sns

pivot = df.pivot_table(
    index='perturbation', columns='model',
    values='rdm_similarity', aggfunc='mean',
)

# Sort columns by model size
col_order = sorted(pivot.columns, key=lambda x: extract_size(x))
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(max(8, len(col_order) * 2.5), 6))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=pivot.values.min() * 0.95, vmax=1.0,
    ax=ax, linewidths=0.5,
)
ax.set_title('RDM Similarity (clean vs. perturbed) by Model and Perturbation',
             fontweight='bold')
ax.set_ylabel('Perturbation')
ax.set_xlabel('Model (small -> large)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/esm2_heatmap_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved heatmap to {RESULTS_DIR}/esm2_heatmap_{PHASE}.png')